# Chapter 18 &mdash; Syntax and the Three Reduction Rules: Alpha, Beta, Eta

**Concept 3 of the Chapter 18 decomposition:** *Syntax and the Three Reduction Rules: Alpha, Beta, Eta*

Variable, abstraction, application &mdash; reduced by renaming, substitution and extensionality.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter18/Concept-Syntax-And-Reduction-Rules/Concept-Syntax-And-Reduction-Rules.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


**Syntax**, in three lines:

$$e ::= x \mid \lambda x.e \mid (e_1\,e_2)$$

a **variable**, an **abstraction** (function), an **application**. Parentheses group;
application associates left, and $\lambda$ extends as far right as possible.

**Three reduction rules:**

* **$\alpha$ (renaming):** $\lambda x.e \equiv \lambda y.e[x:=y]$ for fresh $y$. Bound
  names do not matter.
* **$\beta$ (substitution):** $(\lambda x.e)\,a \to e[x:=a]$. This is *the*
  computation rule.
* **$\eta$ (extensionality):** $\lambda x.(f\,x) \to f$ when $x$ is not free in $f$.
  A function that just passes its argument along **is** that function.

The one hazard is **variable capture**: substituting a term with a free $y$ into a
context that binds $y$. $\alpha$-renaming exists precisely to avoid it.

## 2. Definitions

### A tiny lambda-term representation and beta-reducer

In [ ]:
# terms: ('var', x) | ('lam', x, body) | ('app', f, a)
def V(x):      return ('var', x)
def L(x, b):   return ('lam', x, b)
def A(f, a):   return ('app', f, a)

def show(t):
    if t[0] == 'var': return t[1]
    if t[0] == 'lam': return "(\\%s. %s)" % (t[1], show(t[2]))
    return "(%s %s)" % (show(t[1]), show(t[2]))

def free(t):
    if t[0] == 'var': return {t[1]}
    if t[0] == 'lam': return free(t[2]) - {t[1]}
    return free(t[1]) | free(t[2])

### Capture-avoiding substitution, alpha, beta and eta

In [ ]:
_fresh = [0]
def fresh(base):
    _fresh[0] += 1
    return "%s%d" % (base, _fresh[0])

def subst(t, x, s):
    if t[0] == 'var':
        return s if t[1] == x else t
    if t[0] == 'app':
        return A(subst(t[1], x, s), subst(t[2], x, s))
    y, body = t[1], t[2]
    if y == x: return t                      # x is shadowed here
    if y in free(s):                         # CAPTURE -- alpha-rename first
        z = fresh(y)
        body = subst(body, y, V(z)); y = z
    return L(y, subst(body, x, s))

def alpha(t, newname):
    assert t[0] == 'lam'
    return L(newname, subst(t[2], t[1], V(newname)))

def beta_step(t):
    if t[0] == 'app' and t[1][0] == 'lam':
        return subst(t[1][2], t[1][1], t[2]), True
    if t[0] == 'app':
        f, done = beta_step(t[1])
        if done: return A(f, t[2]), True
        a, done = beta_step(t[2])
        return A(t[1], a), done
    if t[0] == 'lam':
        b, done = beta_step(t[2])
        return L(t[1], b), done
    return t, False

def normalize(t, cap=100):
    steps = [t]
    for _ in range(cap):
        t, done = beta_step(t)
        if not done: break
        steps.append(t)
    return t, steps

def eta(t):
    if t[0] == 'lam' and t[2][0] == 'app' and t[2][2] == V(t[1]) \
       and t[1] not in free(t[2][1]):
        return t[2][1]
    return t

## 3. Tests

The three syntactic forms.

In [ ]:
I = L('x', V('x'))
K = L('x', L('y', V('x')))
app = A(I, V('z'))
for name, t in [('identity', I), ('K', K), ('application', app)]:
    print("  %-12s %s" % (name, show(t)))

**$\beta$:** the computation rule.

In [ ]:
t = A(I, V('z'))
r, steps = normalize(t)
for s in steps: print("   ", show(s))
assert r == V('z')
print()
t2 = A(A(K, V('a')), V('b'))
r2, steps2 = normalize(t2)
for s in steps2: print("   ", show(s))
assert r2 == V('a')

**$\alpha$:** bound names do not matter.

In [ ]:
a1 = L('x', V('x'))
a2 = alpha(a1, 'w')
print("  %s   alpha-renamed to   %s" % (show(a1), show(a2)))
r1, _ = normalize(A(a1, V('q')))
r2, _ = normalize(A(a2, V('q')))
print("  both applied to q give %s and %s" % (show(r1), show(r2)))
assert r1 == r2

**Capture avoidance:** substituting a term with a free `y` under a `y` binder.

In [ ]:
# (\x. \y. x)  applied to  y      must NOT produce  \y. y
t = A(L('x', L('y', V('x'))), V('y'))
r, _ = normalize(t)
print("  term   :", show(t))
print("  reduces:", show(r))
assert r != L('y', V('y')), "capture happened!"
print("\nThe binder was renamed, so the free y stayed free.  Without that,")
print("the constant function would have turned into the identity.")

**$\eta$:** a wrapper that only passes its argument along is redundant.

In [ ]:
f = V('f')
wrapper = L('x', A(f, V('x')))
print("  %s  eta-reduces to  %s" % (show(wrapper), show(eta(wrapper))))
assert eta(wrapper) == f
notok = L('x', A(V('x'), V('x')))
print("  %s  eta-reduces to  %s   (unchanged: x IS free in the function)"
      % (show(notok), show(eta(notok))))
assert eta(notok) == notok

Reduction can also **not terminate** &mdash; Omega.

In [ ]:
omega = L('x', A(V('x'), V('x')))
OMEGA = A(omega, omega)
r, steps = normalize(OMEGA, cap=6)
print("  Omega =", show(OMEGA))
for s in steps[:4]: print("     ->", show(s))
print("  ... unchanged forever")
assert all(s == OMEGA for s in steps)
print("\nSo the lambda-calculus has non-termination, as any universal model must.")

## 4. Exercises


1. Reduce $(\lambda x.\lambda y.\,x\,y)\,(\lambda z.z)\,w$ by hand, then check.
2. Construct a term where capture would occur without $\alpha$-renaming.
3. Is $\eta$ ever *unsound*? (Think about side effects, or about `undefined`.)

In [ ]:
# Your work for the exercises above.